In [ ]:
import sys
import subprocess

required = ["torch", "transformers", "datasets", "scikit-learn"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *required])
print("Installed required packages.")

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
model_name = "textattack/distilbert-base-uncased-MRPC"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
model.to(device)
model.eval()
print(f"Loaded model: {model_name}")
print(f"Number of labels: {model.config.num_labels}")

In [ ]:
dataset = load_dataset("glue", "mrpc", split="validation")
print(f"Dataset split: glue/mrpc validation")
print(f"Number of examples: {len(dataset)}")
print("Example row:")
print(dataset[0])

In [ ]:
batch_size = 32
labels = dataset["label"]
predictions = []
confidences = []

for start_idx in range(0, len(dataset), batch_size):
    batch = dataset[start_idx:start_idx + batch_size]
    inputs = tokenizer(
        batch["sentence1"],
        batch["sentence2"],
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=-1)
        preds = torch.argmax(logits, dim=-1)
    predictions.extend(preds.cpu().tolist())
    confidences.extend(probs.max(dim=-1).values.cpu().tolist())

print(f"Completed inference for {len(predictions)} examples.")

In [ ]:
accuracy = accuracy_score(labels, predictions)
precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average="binary")
cm = confusion_matrix(labels, predictions)

print("Evaluation metrics:")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1       : {f1:.4f}")
print("Confusion matrix:")
print(cm)

In [ ]:
label_map = {0: "not_paraphrase", 1: "paraphrase"}
num_examples_to_show = 5

for i in range(num_examples_to_show):
    row = dataset[i]
    true_label = labels[i]
    pred_label = predictions[i]
    conf = confidences[i]
    print(f"Example {i + 1}")
    print(f"sentence1: {row['sentence1']}")
    print(f"sentence2: {row['sentence2']}")
    print(f"true label: {true_label} ({label_map[true_label]})")
    print(f"pred label: {pred_label} ({label_map[pred_label]})")
    print(f"confidence: {conf:.4f}")
    print("-" * 80)

In [ ]:
print("RESULT SUMMARY")
print(f"model={model_name}")
print("dataset_split=glue/mrpc validation")
print(f"device={device}")
print(f"num_examples={len(dataset)}")
print(f"accuracy={accuracy:.4f}")
print(f"precision={precision:.4f}")
print(f"recall={recall:.4f}")
print(f"f1={f1:.4f}")